# T17 — Fidelity Comparison & Data Masking

Compare real vs synthetic data quality and mask PII in real datasets.

**What you'll learn:**
- Score synthetic data fidelity against real data
- Read markdown fidelity reports
- Mask PII columns automatically
- Configure masking exclusions and overrides

## 1. Generate Real + Synthetic Data

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

real_data = {
    "customer": pd.DataFrame({
        "customer_id": range(1, 1001),
        "name": [f"Person {i}" for i in range(1, 1001)],
        "email": [f"user{i}@company.com" for i in range(1, 1001)],
        "phone": [f"555-{i:04d}" for i in range(1, 1001)],
        "age": rng.normal(40, 12, 1000).astype(int).clip(18, 90),
        "tier": rng.choice(["basic", "silver", "gold"], 1000, p=[0.6, 0.3, 0.1]),
        "spend": rng.lognormal(6, 1.0, 1000).round(2),
    }),
}

# Slightly different synthetic version
rng2 = np.random.default_rng(99)
synth_data = {
    "customer": pd.DataFrame({
        "customer_id": range(1, 1001),
        "name": [f"Synth {i}" for i in range(1, 1001)],
        "email": [f"synth{i}@fake.com" for i in range(1, 1001)],
        "phone": [f"555-{i:04d}" for i in range(1, 1001)],
        "age": rng2.normal(42, 14, 1000).astype(int).clip(18, 90),
        "tier": rng2.choice(["basic", "silver", "gold"], 1000, p=[0.55, 0.30, 0.15]),
        "spend": rng2.lognormal(6.2, 1.1, 1000).round(2),
    }),
}
print(f"Real: {len(real_data['customer'])} rows, Synthetic: {len(synth_data['customer'])} rows")

## 2. Compare Fidelity

In [ ]:
from sqllocks_spindle.inference.comparator import FidelityComparator

comp = FidelityComparator()
report = comp.compare(real_data, synth_data)

print(f"Overall Fidelity Score: {report.overall_score:.1f}/100")
print()
print(report.summary())

## 3. Detailed Column Scores

In [ ]:
for table_name, table_fidelity in report.tables.items():
    print(f"\n=== {table_name} (score: {table_fidelity.score:.1f}) ===")
    for col_name, col_fidelity in table_fidelity.columns.items():
        print(f"  {col_name:20s} score={col_fidelity.score:5.1f}  "
              f"dtype_match={col_fidelity.dtype_match}  "
              f"null_delta={col_fidelity.null_rate_delta:.3f}")

## 4. Markdown Report

In [ ]:
md = report.to_markdown()
print(md[:500])  # First 500 chars

## 5. Mask PII in Real Data

In [ ]:
from sqllocks_spindle.inference.masker import DataMasker, MaskConfig

masker = DataMasker()
mask_result = masker.mask(
    tables=real_data,
    config=MaskConfig(
        seed=42,
        preserve_nulls=True,
        preserve_fks=True,
        exclude_columns=["customer_id", "tier"],
    ),
)

print(f"Masked {mask_result.columns_masked} columns")
print("\nOriginal:")
print(real_data['customer'][['name', 'email', 'phone', 'age']].head())
print("\nMasked:")
print(mask_result.tables['customer'][['name', 'email', 'phone', 'age']].head())

## 6. CLI Equivalent

```bash
# Compare fidelity
spindle compare ./real_data/ ./synthetic/ --format csv --output fidelity.md

# Mask PII
spindle mask ./real_data/ --output ./masked/ --seed 42 --exclude customer_id
```